# Worked solutions · Section 3 of the sample questions

The sample questions are in `documentation/tex/questions`.
For each question the entry gives the answer,
a solution that cites the lecture notes for every formula it uses,
and the misreading that leads to each of the other choices.
The facts that several questions use are stated once, at the head of the section,
as F1–F3.

A citation names the number of a statement in the current build of the notes
and its label, which survives renumbering:
Proposition 1.1.11 (`prop.decompositionOfLimitOrder`), eq. (1.1.10, `eq.vwap`).
Prices are integer counts of ticks, and a traded value is in tick-shares.
A side of a book is written as its occupied levels, price $\times$ size, best first.

The code cells are a calculator: arithmetic on the numbers of the question,
as in the examination.
No snippet is run here;
the notebook `03-vwap` runs them and verifies every key.

In [ ]:
from math import log2

# Section 3 · Traded volume and VWAP

**F1. Fills.**
A buy market order for $Q$ shares takes the asks best first,
$q_i$ shares at the $i$-th level, eq. (1.1.4, `eq.fillAtLevel`),
and each fill trades at the price of the resting order
(the matching rule that precedes eq. (1.1.1, `eq.orderingOfLimitOrders`)).
A buy limit order $(t, q, p, +1)$ is processed as the buy market order for
$$q_M = \min\Big(q, \sum_{i \geq 1} S^{a,i}_{t-}\,\mathbf 1_{\{P^{a,i}_{t-} \leq p\}}\Big)$$
followed by a buy limit order for $q - q_M$ shares resting at $p$,
Proposition 1.1.11 (`prop.decompositionOfLimitOrder`);
a sell mutatis mutandis.
`unito26.lob.orderbook.AggregateBook.submit` is this one code path:
it consumes the opposite side while the best price there satisfies the limit, then rests the remainder.
After a message, `last_traded_size` is the number of shares it traded
and `last_traded_value` the sum over its fills of size times price;
a withdrawal removes resting shares, trades none, and sets both to zero.

**F2. Volume, traded value, VWAP.**
Over the window $(t - w, t]$ the volume $V_{t,w}$ is the sum of the sizes of the market orders,
eq. (1.1.8, `eq.transactedVolume`),
the traded value $\text{TV}_{t,w}$ the sum over their fills of size times price,
eq. (1.1.9, `eq.tradedValue`),
and
$$\text{VWAP}_{t,w} = \frac{\text{TV}_{t,w}}{V_{t,w}},$$
eq. (1.1.10, `eq.vwap`), defined whenever $V_{t,w} > 0$.
It is a ratio of sums: every share traded carries the same weight,
so each order is weighted by its volume.

**F3. Sweep cost.**
For a buy market order for $Q$ shares, with fills $q_i$,
$$\text{sc}_t(Q) = \frac{1}{Q}\sum_{i \geq 1} q_i\big(P^{a,i}_t - P^m_t\big)
= \frac{\phi_t}{2} + \frac{1}{Q}\sum_{i \geq 1} q_i\,(i - 1)\,\tau,$$
eq. (1.1.5, `eq.sweepCost`), with $\tau$ one tick here,
$\phi_t = |P^a_t - P^b_t|$ and $P^m_t = (P^a_t + P^b_t)/2$
(the definitions that precede eq. (1.1.2, `eq.queueImbalance`)).
Since $\sum_i q_i = Q$, the first expression is the VWAP of the fills of that one order less the mid before it.
In both, $i$ runs over the levels of the price grid, $P^{a,i}_t = P^a_t + (i-1)\tau$,
and the second equality follows from $P^a_t - P^m_t = \phi_t/2$ (the sentence after the definition).
An empty level fills nothing, so the first sum is the same over the occupied prices;
in the second, a share filled at a price is charged its distance in ticks from the touch, empty prices between them included.
The two indexings, over the grid and over the occupied prices,
agree only where no gap falls within the levels used,
Remark 1.2.4 (`remark.gridAgainstOccupied`).
The package computes the first expression:
`unito26.lob.orderbook.AggregateBook.sweep_cost` hands `_sweep_cost` the occupied levels best first,
and the cost it computes is `direction * (value / size - mid)`.

## 3.1 · `question.vwapOfABatchAgainstTheBookItLeaves`

Bids $1000 \times 100$, $999 \times 200$; asks $1002 \times 120$, $1003 \times 180$.
A buy market order for 150, then a buy limit order for 100 at 1004.

**Answer: (C).** `1002.52`.

**The first order.**
It takes 120 at 1002 and 30 at 1003 (F1), traded value $150330$,
and leaves the asks $1003 \times 150$.

**The second order.**
It meets the book the first order left.
By Proposition 1.1.11 (`prop.decompositionOfLimitOrder`) (F1) it is the buy market order for
$q_M = \min(100, 150) = 100$, the whole ask side, $1003 \times 150$, lying at or below its limit of 1004,
and nothing rests.
It takes 100 at 1003, at the resting price: traded value $100300$.

**The VWAP.**
$(150330 + 100300)/250 = 1002.52$ (F2).

In [ ]:
first = 120 * 1002 + 30 * 1003
second = 100 * 1003
first, second, (first + second) / 250

**The other choices.**
- (A) `1002.12`: the second order against the initial book, 100 at 1002: $(150330 + 100200)/250$.
- (B) `1002.2`: the limit order taken to rest without trading: $150330/150$.
- (D) `1002.6`: the mean of the two orders' own averages, $(1002.2 + 1003)/2$,
  which weights the orders equally rather than by volume (F2).
- (E) `1002.92`: the second order filled at its own limit price, 1004: $(150330 + 100400)/250$.

In [ ]:
(first + 100 * 1002) / 250, first / 150, (first / 150 + 1003) / 2, (first + 100 * 1004) / 250

## 3.2 · `question.vwapOfTwoBuysAgainstOne`

The book of 3.1, folded afresh for each of four streams:
one buy market order for 200;
two for 100;
two for 100 with a withdrawal of 20 at 1002 on the ask side between them;
two for 100 with a sell limit order for 50 at 1002 between them.

**Answer: (B).** `1002.4`, `1002.4`, `1002.5`, `1002.15`.

- **one**: 120 at 1002 and 80 at 1003, $200480/200 = 1002.4$.
- **two**: the first buy takes 100 at 1002 and leaves 20 there;
  the second takes those 20 and 80 at 1003.
  These are the fills of **one**: $1002.4$.
- **cancelled**: the withdrawal removes the 20 shares left at 1002 and trades nothing (F1),
  so it adds nothing to the value or the volume.
  The second buy then meets $1003 \times 180$ and takes 100 at 1003:
  $(100200 + 100300)/200 = 1002.5$.
- **refilled**: the sell limit order at 1002 faces a best bid of 1000, below its limit,
  so $q_M = 0$ and it rests: the asks become $1002 \times 70$, $1003 \times 180$ (F1).
  The second buy takes 70 at 1002 and 30 at 1003:
  $(100200 + 70140 + 30090)/200 = 1002.15$.

The volume is 200 in every stream;
what differs is the price at which the second buy fills,
which depends on every message that reached the book before it.

In [ ]:
one = (120 * 1002 + 80 * 1003) / 200
cancelled = (100 * 1002 + 100 * 1003) / 200
refilled = (100 * 1002 + 70 * 1002 + 30 * 1003) / 200
one, cancelled, refilled

**The other choices.**
- (A) `1002.4` four times: takes a message that trades nothing to leave the VWAP as it was;
  the withdrawal and the limit sell change the shares the second buy meets at 1002.
- (C) `1002.4`, then `1002.0` three times: each order of a stream filled against the initial book,
  100 at 1002 twice; the book is carried from one message to the next.
- (D) `1002.4`, `1002.4`, `1002.4`, `1002.15`:
  misses that the withdrawal removes the 20 shares at 1002 that the second buy would have taken.

## 3.3 · `question.sweepCostIsTheVwapOfOneOrderLessTheMid`

The book of 3.1, $P^m = 1001$, $\phi = 2$.
The sweep cost of a buy for 160 on the first five occupied ask levels,
the mid read into `before`, the buy for 160 applied,
then its VWAP less `before` and less the mid after it.

**Answer: (A).** `1.25 1.25 0.75`.

**The sweep cost.**
The buy fills 120 at 1002 and 40 at 1003, the first two levels of the grid.
By eq. (1.1.5, `eq.sweepCost`) (F3)
$$\text{sc}(160) = \frac{120 \cdot 1 + 40 \cdot 2}{160} = \frac{\phi}{2} + \frac{40 \cdot 1}{160} = 1.25,$$
larger than $\phi/2 = 1$ because the order walks the book, $160 > S^{a,1} = 120$
(the paragraph on walking the book after Proposition 1.1.11 (`prop.decompositionOfLimitOrder`)).

**The VWAP less the mid before.**
The same order is applied, so its fills are those above:
its VWAP is $(120 \cdot 1002 + 40 \cdot 1003)/160 = 1002.25$,
and $1002.25 - 1001 = 1.25$, the first expression of eq. (1.1.5, `eq.sweepCost`).
`before = book.mid_price` binds the float the property returned, 1001.0,
which the order does not change.

**The VWAP less the mid after.**
The order empties 1002; the asks become $1003 \times 140$, and the mid moves to $(1000 + 1003)/2 = 1001.5$:
$1002.25 - 1001.5 = 0.75$.

In [ ]:
vwap = (120 * 1002 + 40 * 1003) / 160
vwap, 2 / 2 + 40 / 160, vwap - 1001, vwap - (1000 + 1003) / 2

**The other choices.**
- (B) `0.25 1.25 0.75`: the sweep cost taken as the walk term alone, $40/160$,
  measured from the touch rather than from the mid.
- (C) `1.25 1.25 1.25`: takes the order to leave the mid where it was; it empties the best ask.
- (D) `1.25 0.75 0.75`: takes `before` for the property itself, read again after the order;
  it holds the number the property returned.

## 3.4 · `question.sweepCostWalksTheGrid`

Bids $1000 \times 100$; asks $1002 \times 120$, $1004 \times 180$, with 1003 empty;
$P^m = 1001$, $\phi = 2$.
`book.sweep_cost` for a buy of 160, against `walk_cost`,
which transcribes the second expression of eq. (1.1.5, `eq.sweepCost`)
with $i$ the position among the occupied asks.

**Answer: (D).** `1.5 1.25`.

**The package.**
`_sweep_cost` walks the occupied levels $(1002, 120), (1004, 180)$:
120 at 1002 and 40 at 1004, a VWAP of $1002.5$, less the mid: $1.5$ (F3).
On the grid, 1004 is level 3, so eq. (1.1.5, `eq.sweepCost`) gives the same number,
$$\frac{\phi}{2} + \frac{40 \cdot 2}{160} = 1 + 0.5 = 1.5.$$

**`walk_cost`.**
`enumerate(asks, start=1)` numbers the occupied asks:
1004 is $i = 2$, and its 40 shares are charged one tick beyond the touch,
$$\frac{\phi}{2} + \frac{40 \cdot 1}{160} = 1.25.$$
The grid index and the occupied index agree only where no gap falls within the sweep,
Remark 1.2.4 (`remark.gridAgainstOccupied`) (F3);
here 1003 is the gap.

In [ ]:
(120 * 1002 + 40 * 1004) / 160 - 1001, 2 / 2 + 40 * 2 / 160, 2 / 2 + 40 * 1 / 160

**The other choices.**
- (A) `1.5 1.5`: takes `enumerate` over the occupied asks to give 1004 its grid index, 3.
- (B) `1.25 1.25`: takes the package, too, to charge 1004 one tick beyond the touch,
  as though the occupied prices were contiguous; it charges the fill at its price, 1004.
- (C) `0.5 0.25`: the two walk terms alone, $80/160$ and $40/160$, without the half-spread.

## 3.5 · `question.vwapIsARatioOfSums`

Three rows of trades: 300 shares worth $300600$, none, and 100 worth $100000$.
`a` is the ratio of the sums, `b` the mean of the ratios.

**Answer: (E).** `1001.5 1001.0`.

- `a`: the VWAP (F2), $400600/400 = 1001.5$.
- `b`: the ratios are $1002$, $0/0$, which is NaN, and $1000$.
  `mean` skips the NaN and gives the two others the same weight,
  whatever their volumes: $(1002 + 1000)/2 = 1001.0$.

In [ ]:
(300600 + 100000) / (300 + 100), (300600 / 300 + 100000 / 100) / 2

**The other choices.**
- (A) `1001.5 1001.5`: takes the mean of the ratios to weight each row by its volume, as the VWAP does.
- (B) `1001.5 nan`: takes `mean` to propagate the NaN of the empty row; it skips it.
- (C) `1001.0 1001.5`: exchanges the two, taking the ratio of the sums for the unweighted average.
- (D) `1001.5` and a number near $667.3$: counts the empty row as a price of 0, $(1002 + 0 + 1000)/3$.

In [ ]:
(300600 / 300 + 0 + 100000 / 100) / 3

## 3.6 · `question.vwapWindowAtATiedTimestamp`

Four trades at the times $1, 2, 2, 3$, prices $1002, 1003, 1000, 1001$, sizes $100, 100, 300, 100$;
a window of $w = 1$ second.

**Answer: (B).** `[1002.0, 1003.0, 1000.75, 1001.0]`.

**Where each window starts.**
`np.searchsorted(times, times - 1.0, side="right")` counts, for each entry of `times - 1.0`, that is 0, 1, 1, 2,
the times less than or equal to it: `start` is $0, 1, 1, 3$.
A trade exactly $w$ old, as the trade at 1 is for the trades at 2, is outside the window,
which is open on the left as in $(t - w, t]$ of eq. (1.1.8, `eq.transactedVolume`).

**Where each window ends.**
`cumulative[1:]` at row $i$ is the sum of the rows $0, \dots, i$,
so `rolling_sum` returns the sum over the rows `start[i]`, …, $i$:
each window ends at its own row,
and row 1 does not see row 2, although the two share a time.

**The four ratios.**
Row 0: $1002$. Row 1: its own trade, $1003$.
Row 2: rows 1 and 2, $(100300 + 300000)/400 = 1000.75$.
Row 3: its own trade, $1001$.
Rows 0, 2 and 3 are $\text{VWAP}_{t,w}$ of eq. (1.1.10, `eq.vwap`) (F2); row 1 is not:
the window of eq. (1.1.8, `eq.transactedVolume`) is read on the times and takes in both trades at time 2,
so that $\text{VWAP}_{2,1} = 1000.75$. The two agree at the last row of each time.

In [ ]:
(100300 + 300000) / (100 + 300)

**The other choices.**
- (A) `[1002.0, 1000.75, 1000.75, 1001.0]`: the window read on the times, as eq. (1.1.8, `eq.transactedVolume`) reads it,
  so that both rows at time 2 see each other; the code ends each window at its own row.
- (C) `[1002.0, 1002.5, 1001.0, 1000.8]`: the window closed on the left, $[t - w, t]$,
  as `side="left"` would make it: row 1 then takes in row 0, row 2 rows 0 to 2, and row 3 rows 1 to 3.
- (D) `[1002.0, 1001.0, 1001.0, 1000.8]`: both misreadings at once.

In [ ]:
(100200 + 100300) / 200, (100200 + 100300 + 300000) / 500, (100300 + 300000 + 100100) / 500

## 3.7 · `question.rollingVwapByCumulativeSums`

The rolling VWAP at every trade of a session, by `window_start` and `rolling_sum` (A)
and by a walk back over each window (B),
timed at $n = 1000$ and $4000$ trades over 23400 s, with $w = 1800$ s.

**Answer: (A).**
A is faster; from $n = 1000$ to $n = 4000$ its time grows about fourfold or less,
and that of B about sixteenfold.

**B.**
For each of the $n$ rows the `while` loop steps back over every trade of its window.
A window holds a fixed fraction of the day, $1800/23400 = 1/13$,
so about $n/13$ trades, and the work is about $n \cdot n/13$ steps of Python:
quadratic in $n$, sixteenfold from $n = 1000$ to $n = 4000$.

**A.**
`window_start` makes $n$ binary searches of about $\log_2 n$ steps each,
and the two calls of `rolling_sum` one cumulative sum of $n$ entries each, all inside numpy.
The comparisons grow by $4 \log_2 4000 / \log_2 1000 \approx 4.8$ and the cumulative sums by 4;
the fixed cost of the calls of numpy does not grow, and at $n = 1000$ it is of the order of the searches,
so the time grows about fourfold or less.
At $n = 4000$ B makes about $1.2 \times 10^6$ steps of the interpreter,
A about $4000 \log_2 4000 \approx 4.8 \times 10^4$ comparisons in compiled code: A is faster.
The run of `03-vwap` prints a factor of a few thousand.

In [ ]:
1800 / 23400, (4000 / 1000) ** 2, 4000 * log2(4000) / (1000 * log2(1000)), 4000**2 / 13, 4000 * log2(4000)

**The other choices.**
- (B) A faster by a constant factor, each time fourfold:
  takes the trades of a window to stay fixed as $n$ grows; they are a fixed fraction of the session.
- (C) B faster: takes A to sum over the whole session at every row;
  it sums once and differences, `cumulative[1:] - cumulative[start]`.
- (D) A sixteenfold and B fourfold:
  takes a binary search over the session to cost the length of the session rather than its logarithm,
  and the trades of a window, as in (B), to stay fixed as $n$ grows.

## 3.8 · `question.vwapOfAnEmptyWindowIsNan`

Three windows, of traded values $100200, 0, 100300$ and volumes $100, 0, 100$;
mids $1001, 1001, 1001.5$.
The mean, skipping NaN, of the VWAP less the mid, under `np.errstate(invalid="raise")`.

**Answer: (D).**
A raises `FloatingPointError`; B prints `1.25`; C prints a number near $-333$.

- **A**: the arguments of `np.where` are evaluated before it chooses among them,
  so `value / volume` divides 0 by 0 in the empty window,
  an invalid operation, which raises.
- **B**: `np.divide` divides only where `volume > 0`
  and leaves the NaN of `out` in the empty window: the VWAPs are $1002$, NaN, $1003$.
  Subtracting the mid from that NaN is not invalid and gives NaN,
  which `np.nanmean` skips: $(1 + 1.5)/2 = 1.25$.
- **C**: `np.maximum(volume, 1.0)` turns the empty window's $0/0$ into $0/1$:
  a VWAP of 0, which passes for a price 1001 below the mid,
  and the mean is $(1 - 1001 + 1.5)/3 \approx -332.8$.

The VWAP of eq. (1.1.10, `eq.vwap`) is defined only where $V_{t,w} > 0$ (F2):
B leaves it undefined there, and C assigns it a value.

In [ ]:
(1 + 1.5) / 2, (1 - 1001 + 1.5) / 3

**The other choices.**
- (A) All three print `1.25`: takes `np.where` to divide only where it keeps the quotient,
  and C's VWAP of 0 to be skipped as a NaN would be.
- (B) A and B print `1.25`, C near $-333$: takes `np.where` to divide only where `volume > 0`.
- (C) A raises; B and C print `1.25`: takes C's VWAP of 0 for a missing value.
- (E) A and B raise: takes `np.divide` to divide every entry and `where` only to choose among the results,
  as `np.where` does.

## 3.9 · `question.frameSweepCostZeroTimesNan`

Three rows of three ask levels, a buy of 160 against a mid of 1001.
Row 1: $1002 \times 120$, $1003 \times 180$, $1004 \times 50$.
Row 2: $1002 \times 120$, $1003 \times 180$, then a padded level, price NaN and size 0.
Row 3: $1002 \times 120$, then two padded levels.
A keeps `taken * prices` only where `taken > 0`; B sums every product.

**Answer: (C).** A prints `[1.25, 1.25, nan]`; B prints `[1.25, nan, nan]`.

**What is taken.**
`before` is the size resting at the better levels of the row, the cumulative sum less the level's own size,
and `taken` clips 160 less `before` to between 0 and the size of the level:
- row 1: `before` $= (0, 120, 300)$, `taken` $= (120, 40, 0)$: filled;
- row 2: the same, the third level of size 0 taking 0: filled;
- row 3: `before` $= (0, 120, 120)$, `taken` $= (120, 0, 0)$, 120 shares in all: not filled.

**The cost.**
In rows 1 and 2 the fills are 120 at 1002 and 40 at 1003, a VWAP of $1002.25$, and a cost of $1.25$ (F3).
- A drops the products where nothing is taken, so the 0 shares at the padded level of row 2 do not enter:
  `[1.25, 1.25, nan]`, row 3 being NaN because it does not fill.
- B adds $0 \cdot \text{NaN}$, which is NaN, to the value of row 2:
  the NaN reaches a row whose cost is determined,
  and `np.where(filled, cost, np.nan)` keeps it: `[1.25, nan, nan]`.

In [ ]:
min(max(160 - 0, 0), 120), min(max(160 - 120, 0), 180), min(max(160 - 300, 0), 50), (120 * 1002 + 40 * 1003) / 160 - 1001

**The other choices.**
- (A) Both print `[1.25, 1.25, nan]`: takes a level the order never reaches to add nothing to B's sum;
  $0 \cdot \text{NaN}$ is NaN.
- (B) A prints `[1.25, 1.25, 1.0]` and B `[1.25, nan, nan]`:
  divides the third row's value by the 120 shares it takes rather than by `size`, `filled` disregarded: $1002 - 1001 = 1$;
  in B the $0 \cdot \text{NaN}$ of its padded levels makes that row NaN in any case.
  `filled` is false there, and both return NaN.
- (D) Both print `[1.25, nan, nan]`: takes A to sum every product, $0 \cdot \text{NaN}$ included, as B does;
  its `np.where(taken > 0, ...)` puts 0 in place of the products where no share is taken.

## 3.10 · `question.lobsterAggressorIsMinusDirection`

Six rows of a LOBSTER message file; the executions (type 4) are
96 at 1002 with `Direction` $-1$, 60 at 1000 with $+1$, 32 at 1003 with $-1$, and 20 at 999 with $+1$.
A signs the aggressor $-$`Direction`, B `Direction`.

**Answer: (B).** A prints `1002.25 999.75`; B prints `999.75 1002.25`.

**The aggressor.**
A trade is recorded as the execution of the resting order it matched,
so the direction of an execution names the passive side and the aggressor's direction is its opposite,
Remark 1.2.5 (`remark.passiveDirection`).
The executed sells at 1002 and 1003 were bought by market orders;
the executed buys at 1000 and 999 were sold into.

**The two VWAPs** (F2).
`size` is zero on the rows that are not executions, which therefore add nothing to either sum.
- A: the buyer-initiated trades are 96 at 1002 and 32 at 1003, $(96 \cdot 1002 + 32 \cdot 1003)/128 = 1002.25$;
  the seller-initiated ones 60 at 1000 and 20 at 999, $(60 \cdot 1000 + 20 \cdot 999)/80 = 999.75$.
- B: `buy` is A's `~buy` on every execution, so the two VWAPs change places:
  its buyers pay the bid.

In [ ]:
(96 * 1002 + 32 * 1003) / 128, (60 * 1000 + 20 * 999) / 80

**The other choices.**
- (A) Both print `1002.25 999.75`: takes B to negate `Direction` as A does;
  the two sign every execution oppositely.
- (C) A prints `999.75 1002.25` and B `1002.25 999.75`:
  reads `buy = aggressor > 0` as selecting the rows whose aggressor is negative, in both,
  so that each prints its two VWAPs in the other order.
- (D) Both print `999.75 1002.25`: overlooks the minus sign of A, which then computes what B does.